In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define dataset and output paths
fcm_path = f'{project_path}/outputs/fcm_datasets'
results_path = f'{project_path}/outputs/fcm_results'

In [ ]:
# STEP 1: Extract ResNet50 features of each body part and concatenate them
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

body_parts = ['body', 'front-leg', 'back-leg', 'tail']


def extract_features(test_path):
    resnet = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(180, 180, 3), pooling='avg')
    data, cats = {}, None
    for split in ['train', 'val', 'test']:
        features, labels = [], None
        for part in body_parts:
            ds = tf.keras.utils.image_dataset_from_directory(
                f'{test_path}/{part}/{split}', image_size=(180, 180), batch_size=32, shuffle=False)
            y = np.concatenate([batch.numpy() for _, batch in ds])
            assert cats in (None, ds.class_names), f'{part}/{split} has different cat folders'
            assert labels is None or np.array_equal(labels, y), f'{part}/{split} is not balanced'
            cats, labels = ds.class_names, y
            features.append(resnet.predict(ds, verbose=0))
        data[split] = (np.concatenate(features, axis=-1), labels)
    return data, len(cats)

In [ ]:
# STEP 2: Train and test the FCM, 5 runs per batch size and epoch
def train_and_test(data, n_cats, batch_size, epochs):
    tf.keras.backend.clear_session()
    gc.collect()
    model = tf.keras.Sequential([
        tf.keras.Input(shape=data['train'][0].shape[1:]),
        tf.keras.layers.Dense(n_cats, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(*data['train'], validation_data=data['val'], epochs=epochs, batch_size=batch_size, verbose=0)
    loss, accuracy = model.evaluate(*data['test'], batch_size=batch_size, verbose=0)
    return accuracy, loss


results = []
for test in ['test1', 'test2', 'test3a', 'test3b', 'test4']:
    data, n_cats = extract_features(f'{fcm_path}/{test}')
    for batch_size in [32, 16, 8, 4, 2, 1]:
        for epochs in [10, 20, 30]:
            runs = [train_and_test(data, n_cats, batch_size, epochs) for _ in range(5)]
            accuracy, loss = np.mean(runs, axis=0)
            results.append([test, batch_size, epochs, round(accuracy, 3), round(loss, 3)])
            print(f'{test}  batch_size={batch_size}  epochs={epochs}  accuracy={accuracy:.3f}  loss={loss:.3f}')

In [ ]:
# STEP 3: Save results
results = pd.DataFrame(results, columns=['test', 'batch_size', 'epochs', 'test_accuracy_avg', 'test_loss_avg'])
Path(results_path).mkdir(parents=True, exist_ok=True)
results.to_csv(f'{results_path}/results.csv', index=False)
results.groupby('test', sort=False)[['test_accuracy_avg', 'test_loss_avg']].mean().round(3)